In [2]:
import shutil
from pathlib import Path
from prefect import task, flow, get_run_logger

In [3]:
@task(retries=2, retry_delay_seconds=5)
def verificar_arquivos_locais(arquivos: dict, diretorio_base: Path):
    """Valida se os arquivos sintéticos foram gerados corretamente na camada raw."""
    logger = get_run_logger()
    logger.info("Iniciando validação de integridade dos arquivos locais (Camada Raw)...")
    
    for arquivo in arquivos.keys():
        caminho_completo = diretorio_base / arquivo
        if not caminho_completo.exists():
            raise FileNotFoundError(f"Arquivo ausente: {caminho_completo}")
    logger.info("Todos os arquivos sintéticos foram localizados com sucesso.")

In [4]:
@task
def garantir_infraestrutura_mock(bucket_name: str, base_path: Path) -> Path:
    """Cria a estrutura de pastas local que simula o Google Cloud Storage."""
    logger = get_run_logger()
    
    # Define o caminho do bucket falso dentro da pasta 'data'
    mock_bucket_path = base_path / "data" / "gcp_storage_mock" / bucket_name
    
    if not mock_bucket_path.exists():
        logger.warning(f"Mock Bucket não encontrado. Criando ambiente simulado '{bucket_name}'...")
        mock_bucket_path.mkdir(parents=True, exist_ok=True)
        logger.info(f"Mock Bucket criado com sucesso em: {mock_bucket_path}")
    else:
        logger.info(f"Mock Bucket '{bucket_name}' já existe e está pronto para receber os dados.")
        
    return mock_bucket_path

In [5]:
@task
def executar_upload_mock(local_path: Path, mock_bucket_path: Path, destination_blob: str):
    """Simula o tráfego de rede copiando o arquivo para o Mock Bucket."""
    logger = get_run_logger()
    logger.info(f"Simulando upload de {local_path.name} para gs://{mock_bucket_path.name}/{destination_blob} ...")
    
    # Cria os subdiretórios no destino caso não existam
    destination_path = mock_bucket_path / destination_blob
    destination_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Copia o arquivo fisicamente simulando a carga pra nuvem
    shutil.copy(str(local_path), str(destination_path))
    logger.info(f"Upload simulado de {local_path.name} concluído com sucesso.")

In [7]:
import os
from pathlib import Path

@flow(name="Aposta Ganha - Ingestão Mock")
def iniciar_pipeline_mock():
    """Flow principal que orquestra as tarefas sem depender de faturamento do GCP."""
    logger = get_run_logger()
    logger.info("Iniciando o motor de orquestração do pipeline de contingência...")

    # Configurações do ambiente simulado
    BUCKET_NAME = "aposta_ganha_raw_data_lake" 
    
    # Resolve os caminhos dinâmicos a partir do diretório onde o notebook está rodando
    notebook_dir = Path(".").resolve()
    local_raw_dir = notebook_dir / "data" / "raw"
    
    # Mapeamento dos arquivos gerados na etapa anterior
    files_to_process = {
        "dim_usuarios.csv": "raw/dim_usuarios.csv",
        "fato_transacoes.csv": "raw/fato_transacoes.csv"
    }

    # 1. Validação
    verificar_arquivos_locais(arquivos=files_to_process, diretorio_base=local_raw_dir)
    
    # 2. Infraestrutura (Falsa)
    target_bucket_path = garantir_infraestrutura_mock(bucket_name=BUCKET_NAME, base_path=notebook_dir)

    # 3. Upload (Cópia Local)
    for local_name, cloud_path in files_to_process.items():
        executar_upload_mock(
            local_path=local_raw_dir / local_name, 
            mock_bucket_path=target_bucket_path,
            destination_blob=cloud_path
        )
        
    logger.info("O pipeline foi finalizado e os dados estão salvos na estrutura de Cloud Simulada!")

# Dispara a orquestração dentro do Notebook
# Ajusta o diretório de trabalho para a pasta onde a camada raw realmente existe.
current = Path(".").resolve()
for root in (current, current.parent, current.parent.parent):
    if (root / "data" / "raw").exists():
        os.chdir(root)
        break
else:
    os.chdir(current)

iniciar_pipeline_mock()

13:24:37.081 | INFO    | Flow run 'skilled-seriema' - Beginning flow run 'skilled-seriema' for flow 'Aposta Ganha - Ingestão Mock'

13:24:37.085 | INFO    | Flow run 'skilled-seriema' - Iniciando o motor de orquestração do pipeline de contingência...

13:24:37.098 | INFO    | Task run 'verificar_arquivos_locais-df7' - Iniciando validação de integridade dos arquivos locais (Camada Raw)...

13:24:37.115 | INFO    | Task run 'verificar_arquivos_locais-df7' - Todos os arquivos sintéticos foram localizados com sucesso.

13:24:37.128 | INFO    | Task run 'verificar_arquivos_locais-df7' - Finished in state Completed()

...

13:24:37.304 | INFO    | Task run 'executar_upload_mock-d7d' - Simulando upload de fato_transacoes.csv para gs://aposta_ganha_raw_data_lake/raw/fato_transacoes.csv ...

13:24:37.354 | INFO    | Task run 'executar_upload_mock-d7d' - Upload simulado de fato_transacoes.csv concluído com sucesso.

13:24:37.366 | INFO    | Task run 'executar_upload_mock-d7d' - Finished in state Completed()

13:24:37.371 | INFO    | Flow run 'skilled-seriema' - O pipeline foi finalizado e os dados estão salvos na estrutura de Cloud Simulada!

13:24:38.086 | INFO    | Flow run 'skilled-seriema' - Finished in state Completed()